<a href="https://colab.research.google.com/github/sudikshyapant/lgmd_for_retinal_fundus/blob/main/lgmd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LGMD — Concept Discovery on Retinal Fundus Images

Post-hoc concept discovery for a **5-grade diabetic-retinopathy classifier**, via
*Language-Guided Matrix Decomposition* (LGMD, ECCV 2026 #11398).

Scope: one DR severity grade at a time (the active class). The backbone is **RETFound
(MAE ViT-L/16)** used as a **frozen** foundation encoder with a **linear DR-grading head**
probed on top (the encoder is not fine-tuned). Grades: **0 No DR, 1 Mild NPDR, 2 Moderate
NPDR, 3 Severe NPDR, 4 Proliferative DR**. Concept maps come from **FLAIR** (a retinal CLIP).
Baselines (**ICE, CRAFT, FACE**) and the metrics (**Acc, C-Ins, MSE**) follow the paper.
Concepts are read from `concept_vocab.json` (curated per grade, used verbatim). Heavy
artifacts are cached.

### Scope

The cells below walk through the pipeline on a **single grade** (the active class). To run
**all 5 grades**, jump to the last section — it loads FLAIR once and loops
`config.select_class` over every grade, caching per grade. A grade that errors is printed and
skipped. Qualitative overlays are saved only for `config.FIGURE_CLASSES` (the lesion-rich
grades 2/3/4). A separate section reports FLAIR **concept grounding** (table + figure).

In [ ]:
# === Colab setup (skip when running locally) ===
# 1) Clone the repo from GitHub and install dependencies:
# !git clone https://github.com/sudikshyapant/lgmd_for_retinal_fundus
# %cd lgmd_for_retinal_fundus
# !pip -q install -r requirements.txt
#
# 2) Models: FLAIR (retinal CLIP) downloads public weights on first use. RETFound (MAE
#    ViT-L/16) builds its architecture via timm; its pretrained encoder weights are NOT
#    public here — place the checkpoint at CONFIG["retfound_weights"] (see section 1).
#    config.get_secret still supports HF_TOKEN via Colab Secrets if needed.
# 3) Importing config below mounts Google Drive automatically when on Colab.

In [1]:
# !git clone https://github.com/sudikshyapant/lgmd_for_retinal_fundus

Cloning into 'lgmd_for_retinal_fundus'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 90 (delta 38), reused 73 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (90/90), 8.71 MiB | 34.43 MiB/s, done.
Resolving deltas: 100% (38/38), done.


In [2]:
# %cd lgmd_for_retinal_fundus

/content/lgmd_for_retinal_fundus


In [3]:
!pip -q install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.8 MB/s eta 0:00:00


In [5]:
import os, json, sys

# This notebook belongs to the lgmd_for_retinal_fundus repo. A sibling repo may be cloned
# alongside it and ALSO have a src/config.py, so we anchor on a marker unique to this repo
# (src/lgmd.py) to avoid importing the wrong config. We locate the repo root regardless of
# where the kernel started, cd into it (so config's REPO_ROOT-relative paths resolve), and
# put its src/ first on sys.path.
_MARKERS = (os.path.join("src", "config.py"), os.path.join("src", "lgmd.py"))
_here = os.getcwd()
_candidates = [
    _here,
    os.path.join(_here, "lgmd_for_retinal_fundus"),
    "/content/lgmd_for_retinal_fundus",
    os.path.dirname(_here),
]
for _root in _candidates:
    if all(os.path.isfile(os.path.join(_root, m)) for m in _MARKERS):
        os.chdir(_root)
        sys.path.insert(0, os.path.join(_root, "src"))
        break
else:
    raise RuntimeError(
        "Could not locate the repo root (needs src/config.py and src/lgmd.py). "
        "On Colab, %cd /content/lgmd_for_retinal_fundus first."
    )

import torch
import config
from config import CONFIG, cache_name
import utils, data_utils, model_utils, concepts as concept_mod, flair_maps, lgmd, baselines, metrics, viz

utils.set_seed(CONFIG["seed"])
DEVICE = model_utils.DEVICE
CDIR = CONFIG["cache_dir"]       # heavy/constant artifacts (activations, S, W)
RDIR = CONFIG["results_dir"]     # cached metric tables
bb = CONFIG["backbone"]          # used in figure filenames
print("device:", DEVICE)

Mounted at /content/drive
device: cuda


In [ ]:
# === Artifact namespace toggle — run this right after setup, before any heavy cell ===
# Chooses which set of cached artifacts + figures this session reads/writes:
#   "dr_grading_run1" -> a fresh, isolated namespace (the config.py default)
#   ""                -> the ORIGINAL pre-tag artifacts (switch back to an old run)
# The tag is mixed into every cache filename and prefixes every saved figure, so the two
# namespaces never overwrite each other. Flip it and re-run the pipeline / figure / table
# cells to move between runs. (config.py is intentionally not reloaded elsewhere, so setting
# CONFIG["run_tag"] here in-place is what actually takes effect on a running kernel.)
CONFIG["run_tag"] = "dr_grading_run1"   # <- set to "" to reuse the old artifacts
print("Active run_tag:", CONFIG["run_tag"] or "(original / pre-tag artifacts)")

### A note on hyperparameters

A few values are best-guess estimates, not paper-specified: red-circle radius and stroke
width, PGD iteration counts, CRAFT depth, FACE settings, and the choice of C-Ins metric.
They are marked `[suppl]` in `config.py` because the supplementary material describes them
only qualitatively. The CLIP red-circle prompting setup (A1.6) is followed exactly; only the
items listed above are estimates. Concepts are taken verbatim from the curated
`concept_vocab.json` (no LLM generation and no lexical/CLIP filtering).

## 0. Dataset — IDRiD Disease Grading, 5 grades (download + prepare)

The classifier is trained on **IDRiD** (Indian Diabetic Retinopathy Image Dataset),
**Disease Grading** part: image-level DR grade 0–4. IDRiD ships as flat image folders + label
CSVs, **not** an ImageFolder tree, so `dr_prep.prepare` converts it once: it reads the
`Retinopathy grade` column and writes the grade folders `0_no_dr … 4_proliferative_dr`. IDRiD
has no official validation set, so its **Testing Set** becomes `test/` and its **Training
Set** is stratified **80/20** into `train/` + `val/` (per grade).

The cell below downloads IDRiD via `kagglehub` (cached) and runs the conversion. Credentials
(`KAGGLE_USERNAME`, `KAGGLE_KEY`) come from `config.get_secret` (Colab Secrets → env vars →
gitignored `secrets.json`). Note the grades are imbalanced (Mild NPDR is scarce).

In [6]:
# === Dataset: IDRiD Disease Grading -> 5-grade train/val/test ImageFolder ===
# Source: https://www.kaggle.com/datasets/mariaherrerot/idrid-dataset
#
# kagglehub pulls IDRiD once and caches it; dr_prep.prepare() then builds the grade
# ImageFolder the rest of the pipeline reads and sets CONFIG["data_root"]. Credentials come
# from config.get_secret (Colab Secrets -> env vars -> a gitignored secrets.json), looking
# for KAGGLE_USERNAME and KAGGLE_KEY. Create a token at kaggle.com -> Settings -> "Create New
# Token". If your IDRiD copy uses a different slug/path, pass dr_prep.prepare(slug=...) or
# dr_prep.prepare(raw_root=<your folder>).
import os, config, dr_prep
from config import CONFIG

for _var in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
    if _var not in os.environ:
        try:
            os.environ[_var] = config.get_secret(_var)
        except KeyError:
            pass
os.environ.setdefault("KAGGLEHUB_CACHE", os.path.join(config.BASE_DIR, "kagglehub"))

dr_prep.prepare()   # downloads IDRiD, builds ImageFolder, sets CONFIG["data_root"]
print("data_root :", CONFIG["data_root"])
print("grades    :", config.fundus_class_names(refresh=True))

100%|██████████| 1.62G/1.62G [00:11<00:00, 154MB/s]

Extracting files...


[odir_prep] label table: /content/drive/MyDrive/lgmd/kagglehub/datasets/andrewmvd/ocular-disease-recognition-odir5k/versions/2/full_df.csv
[odir_prep] indexed 8000 images under /content/drive/MyDrive/lgmd/kagglehub/datasets/andrewmvd/ocular-disease-recognition-odir5k/versions/2
[odir_prep] built /content/drive/MyDrive/lgmd/cache/odir5k_imagefolder: 5324 images across 5 classes
    AMD        train= 186 val= 40 test= 40 (total 266)
    Cataract   train= 205 val= 44 test= 44 (total 293)
    Diabetes   train=1126 val=241 test=241 (total 1608)
    Glaucoma   train= 199 val= 43 test= 42 (total 284)
    Normal     train=2011 val=431 test=431 (total 2873)
data_root : /content/drive/MyDrive/lgmd/cache/odir5k_imagefolder
classes   : ['AMD', 'Cataract', 'Diabetes', 'Glaucoma', 'Normal']


## 1. Train the DR-grading head (linear probe, run once)

LGMD explains a *trained* classifier, so first attach a head to RETFound. RETFound's MAE
ViT-L/16 encoder is **frozen**; only a `Linear(1024 → 5)` head is trained on the grade labels
(a linear probe — fast, CPU-feasible). Requires the RETFound encoder checkpoint at
`CONFIG["retfound_weights"]`. This caches the trained head (`retfound_mae_head_dr.pt`); rerun
only when you change the data or the training knobs. Everything below loads it.

In [ ]:
# === RETFound MAE encoder weights (download once, guarded) ===
# The frozen RETFound encoder is NOT public in this repo; fetch it from the official gated
# Hugging Face repo (request access once via https://github.com/rmaphoh/RETFound_MAE). This
# skips the download when the checkpoint is already at CONFIG["retfound_weights"] (cached on
# Drive), so it's safe to re-run every session.
import os, shutil, config
from config import CONFIG

# Confirm these against the RETFound_MAE README — the CFP = color-fundus MAE encoder (IDRiD is
# color fundus, NOT OCT). Edit them if you were granted access under a different repo id/file.
RETFOUND_HF_REPO = "YukunZhou/RETFound_mae_natureCFP"
RETFOUND_HF_FILE = "RETFound_mae_natureCFP.pth"

if os.path.exists(CONFIG["retfound_weights"]):
    print("RETFound encoder already present:", CONFIG["retfound_weights"])
else:
    from huggingface_hub import login, hf_hub_download
    login(config.get_secret("HF_TOKEN"))          # HF token: Colab Secrets / env / secrets.json
    src = hf_hub_download(repo_id=RETFOUND_HF_REPO, filename=RETFOUND_HF_FILE)
    os.makedirs(os.path.dirname(CONFIG["retfound_weights"]), exist_ok=True)
    shutil.copy(src, CONFIG["retfound_weights"])
    print("downloaded RETFound encoder ->", CONFIG["retfound_weights"])

In [ ]:
import train_backbone
from config import CONFIG

# One-time linear probe of the DR-grading head (RETFound encoder frozen). Needs the
# RETFound (MAE ViT-L/16) encoder checkpoint at CONFIG["retfound_weights"].
if not os.path.exists(CONFIG["retfound_weights"]):
    raise FileNotFoundError(
        f"RETFound encoder checkpoint not found at {CONFIG['retfound_weights']}. Download the "
        f"RETFound MAE (natureCFP) weights and place them there (or update the path) first.")
if not os.path.exists(CONFIG["backbone_weights"]):
    train_backbone.train()
else:
    print("trained head already cached:", CONFIG["backbone_weights"])

## 2. Data — the active fundus class

In [1]:
print("Fundus grades:", config.fundus_class_names())
idx = config.select_class(CONFIG["class_name"])   # active grade (default: 2_moderate_npdr)
print(f"Active grade: {CONFIG['class_name']!r}  (label index {idx})")

NameError: name 'config' is not defined

In [ ]:
cls, seed = CONFIG["class_name"], CONFIG["seed"]
train_imgs = data_utils.load_class_images(cls, CONFIG["n_train"], "train_dir", seed)
val_imgs   = data_utils.load_class_images(cls, CONFIG["n_val"],   "val_dir",   seed)
print(f"{cls}: train {len(train_imgs)}  val {len(val_imgs)}")

## 3. Backbone + encoder activations (cached)

In [ ]:
# Backbone and FLAIR share this single 224x224 crop on purpose. Reading the same pixels
# makes their 7x7 spatial grids line up cell-for-cell, so the concept maps stay spatially
# accurate. That is why preprocessing is one shared step here, not separate per-model crops.

model, transform = model_utils.load_backbone()
Z_train = utils.cached(os.path.join(CDIR, cache_name("Z_train", ".pt", "data", "model")),
    lambda: model_utils.extract_activations(model, transform, train_imgs, desc="train activations"))
Z_val = utils.cached(os.path.join(CDIR, cache_name("Z_val", ".pt", "data", "model")),
    lambda: model_utils.extract_activations(model, transform, val_imgs, desc="val activations"))

# RETFound is a ViT: its patch tokens are signed, but NMF needs non-negative input. Shift the
# activations into the non-negative orthant by their per-channel min (from train); every
# method below works in this shifted space, and lgmd.reconstruct adds the offset back so the
# classifier head still reads original-space features (predictions unchanged).
A_train = lgmd.unfold(Z_train)
offset = A_train.min(dim=0).values          # (p,) per-channel shift, shared by all methods
A_train = A_train - offset                  # non-negative feature space
A_val = lgmd.unfold(Z_val) - offset
print(bb, "| Z_train:", tuple(Z_train.shape), " A_train:", tuple(A_train.shape))

## 4. Concept vocabulary (stored table, curated per grade)

Concepts are read from the static `concept_vocab.json` table (curated **per grade**, grouped
by lesion category) and used **verbatim** — every listed concept becomes a basis column, in
order, with variable count per grade. Reproducible and offline — no LLM call.

In [ ]:
# get_concepts loads this grade's curated concepts verbatim: every listed concept becomes a
# basis column, in order (no lexical/CLIP reduction, no LLM). FLAIR is the retinal CLIP behind
# the maps S; it is passed for pipeline symmetry but is not used to filter concepts here.

vlm = flair_maps.VLM()
concept_list = concept_mod.get_concepts(vlm, images=train_imgs)
print(f"{len(concept_list)} concepts:", concept_list)

## 5. Localized CLIP similarity maps S (cached — most expensive stage)

In [ ]:
S_train = utils.cached(os.path.join(CDIR, cache_name("S_train", ".pt", "data", "con", "clip")),
    lambda: flair_maps.build_S(train_imgs, concept_list, vlm))
print("S_train:", tuple(S_train.shape))

## 6. Fit the semantic concept basis W (PGD)

In [ ]:
W = utils.cached(os.path.join(CDIR, cache_name("W", ".pt", "data", "model", "con", "clip", "pgd")),
    lambda: lgmd.fit_basis(A_train, S_train))
print("W:", tuple(W.shape))

## 7. Inference on val + predictive preservation / faithfulness

In [ ]:
label = CONFIG["class_index"]

# Paper Sec 4: concept analysis is performed on correctly classified validation samples.
orig_logits_full = model_utils.logits_from_Z(model, Z_val)   # raw (original-space) activations
keep = orig_logits_full.argmax(-1) == label
print(f"correctly classified val: {int(keep.sum())}/{len(keep)}")
Z_val = Z_val[keep]
val_imgs = [im for im, k in zip(val_imgs, keep.tolist()) if k]
A_val = lgmd.unfold(Z_val) - offset              # shift into the non-negative feature space
orig_logits = orig_logits_full[keep]

S_hat = lgmd.infer(A_val, W)                      # closed-form + PGD (Eqs. 4-5)
A_hat = lgmd.reconstruct(S_hat, W, Z_val.shape, offset)   # offset added back -> original space

def _lgmd_metrics():
    recon_logits = model_utils.logits_from_Z(model, A_hat)
    return {
        **metrics.predictive_preservation(orig_logits, recon_logits, label),
        "kl": metrics.kl_logits(orig_logits, recon_logits),
        "recon_err": metrics.recon_error(A_val, lgmd.unfold(A_hat) - offset),  # measured in shifted space
    }

lgmd_metrics = utils.cached_json(
    os.path.join(RDIR, cache_name("lgmd_metrics", ".json", "data", "model", "con", "clip", "pgd", "infer")),
    _lgmd_metrics)
print("LGMD:", lgmd_metrics)

## 8. Baseline comparison (ICE, CRAFT, FACE vs LGMD) — Acc + C-Ins

In [ ]:
R = len(concept_list)   # basis columns = number of concepts
# face_head adds the offset back so FACE's KL term aligns with the ORIGINAL classifier logits;
# head_fn stays plain because reconstructions are handed to it already in original space.
offset_dev = offset.to(DEVICE)
face_head = lambda a: model_utils.classify_pooled(model, a.to(DEVICE) + offset_dev)
head_fn = lambda Z: model_utils.logits_from_Z(model, Z)

def _comparison():
    # ICE, CRAFT, and FACE differ only in how the basis W is learned (NMF, recursive
    # NMF, and KL-regularized NMF respectively). Everything else is held identical:
    # backbone, preprocessing, splits, concept count, non-negative shift, and inference.
    bases = {
        "ICE":   baselines.fit_ice(A_train, R),
        "CRAFT": baselines.fit_craft(A_train, R),
        "FACE":  baselines.fit_face(A_train, R, face_head, Z_train.shape),
        "LGMD":  W,
    }
    out = {}
    for name, Wb in bases.items():
        Sb = lgmd.infer(A_val, Wb)
        Ab = lgmd.reconstruct(Sb, Wb, Z_val.shape, offset)   # original space
        lg = model_utils.logits_from_Z(model, Ab)
        cur = metrics.insertion_curve(Sb, Wb, Z_val.shape, head_fn, label, offset=offset)
        pp = metrics.predictive_preservation(orig_logits, lg, label)
        out[name] = {
            "Acc": pp["recon_acc"],
            # C-Ins is the normalized area under the concept-insertion curve (Sec 4.2):
            # how quickly the correct-class prediction is restored as top-ranked
            # concepts are added back in.
            "C-Ins": metrics.insertion_auc(cur),
            "agreement": pp["agreement"],
            "kl": metrics.kl_logits(orig_logits, lg),
            "recon_err": metrics.recon_error(A_val, lgmd.unfold(Ab) - offset),
        }
    return out

rows = utils.cached_json(
    os.path.join(RDIR, cache_name("comparison", ".json",
                                  "data", "model", "con", "clip", "pgd", "infer", "base", "cins")),
    _comparison)

# Print the results table
import pandas as pd
df_comp = pd.DataFrame(rows).T
display(df_comp)
print(json.dumps(rows, indent=2))

## 9. Concept-Insertion curve for LGMD

In [ ]:
import matplotlib.pyplot as plt
curve = metrics.insertion_curve(S_hat, W, Z_val.shape, head_fn, label, offset=offset)
plt.plot(curve, label="C-Insertion")
plt.xlabel("# concepts added"); plt.ylabel("true-class prob"); plt.legend()
plt.savefig(os.path.join(RDIR, f"insertion_{bb}.png"), dpi=120, bbox_inches="tight")
plt.show()

## 10. Concept heatmap overlays (Eq. 6)

In [ ]:
viz.save_concept_overlays(val_imgs, S_hat, concept_list, out_name=f"overlays_{bb}.png", max_images=4)

## 11. Ablation — closed-form vs closed-form+PGD coefficient estimation (Table 4)

In [ ]:
# Closed-form (Eq. 4) vs closed-form + PGD (Eq. 5). Both preserve accuracy; the
# hybrid slightly improves C-Ins and lowers reconstruction MSE (paper Sec 4.5). Cached on Drive.
def _ablation():
    out = {}
    for mode, refine in [("Closed-form", False), ("Closed-form+PGD", True)]:
        Sh = lgmd.infer(A_val, W, refine=refine)
        Ah = lgmd.reconstruct(Sh, W, Z_val.shape, offset)
        lg = model_utils.logits_from_Z(model, Ah)
        pp = metrics.predictive_preservation(orig_logits, lg, label)
        cur = metrics.insertion_curve(Sh, W, Z_val.shape, head_fn, label, offset=offset)
        out[mode] = {
            "Acc": pp["recon_acc"],
            "C-Ins": metrics.insertion_auc(cur),
            "MSE": metrics.mse(A_val, lgmd.unfold(Ah) - offset),
        }
    return out

abl = utils.cached_json(
    os.path.join(RDIR, cache_name("ablation_table4", ".json",
                                  "data", "model", "con", "clip", "pgd", "infer", "cins")),
    _ablation)
print(json.dumps(abl, indent=2))

## 12. Concept activation scores: before (CLIP init) vs after optimization (Fig 4)

In [ ]:
import importlib
import viz

# Reload the viz module to apply changes from the disk
importlib.reload(viz)
print("Successfully reloaded viz.py")

In [ ]:
# "Before" = raw CLIP-similarity init on val images (expensive: cached); "after" = S_hat.
# Depends on backbone too: val images are filtered to correctly classified samples.
S_val_vlm = utils.cached(os.path.join(CDIR, cache_name("S_val", ".pt", "data", "model", "con", "clip")),
    lambda: flair_maps.build_S(val_imgs, concept_list, vlm))   # same row order as A_val / S_hat
viz.plot_score_distributions(S_val_vlm, S_hat, out_name=f"fig4_score_distributions_{bb}.png")


In [ ]:
# Single clean row: input image + its top named-concept heatmaps.
viz.plot_concept_heatmaps(val_imgs, S_hat, concept_list, img_index=0, top_k=3,
                          out_name=f"fig1_heatmaps_{bb}.png")

In [ ]:
# Fig 3 (example-patch style): each concept shown by its top-activating image regions.
# ICE localizes on the pre-pool feature map -> red region outlines ('red circle regions');
# CRAFT/FACE pool spatially -> cropped exemplar patches; LGMD = cropped patches + names.
# Re-fits the baseline bases (expensive) so the cell is self-contained.
import lgmd
bases = {
    "ICE":   baselines.fit_ice(A_train, R),
    "CRAFT": baselines.fit_craft(A_train, R),
    "FACE":  baselines.fit_face(A_train, R, face_head, Z_train.shape),
    "LGMD":  W,
}
method_maps = {name: lgmd.infer(A_val, Wb) for name, Wb in bases.items()}
styles = {"ICE": "contour", "CRAFT": "crop", "FACE": "crop", "LGMD": "crop"}
for name, S in method_maps.items():
    viz.plot_concept_patches(
        val_imgs, S,
        concepts=concept_list if name == "LGMD" else None,
        style=styles[name], top_concepts=3, n_patches=3, title=name,
        out_name=f"fig3_{name.lower()}_{bb}.png")

In [ ]:
bases = {
    "ICE":   baselines.fit_ice(A_train, R),
    "CRAFT": baselines.fit_craft(A_train, R),
    "FACE":  baselines.fit_face(A_train, R, face_head, Z_train.shape),
    "LGMD":  W,
}
method_maps = {name: lgmd.infer(A_val, Wb) for name, Wb in bases.items()}
viz.plot_baseline_comparison(val_imgs, method_maps, n_images=3,
                             concepts_by_method={"LGMD": concept_list},
                             out_name=f"fig3_baselines_{bb}.png")

## 13. Run all 5 DR grades

Everything above runs the active single grade. This driver runs all grades: it loads the
backbone + FLAIR once, then loops over `config.fundus_class_names()`, caching every artifact
per grade. A grade that errors is printed and skipped. Concept-overlay figures are produced
only for `config.FIGURE_CLASSES` (grades 2/3/4). Pass a subset to `run_all([...])` for fewer.
A scarce grade (e.g. Mild NPDR) whose head diagnoses too few val images correctly falls back
to evaluating on all its val images (`CONFIG["min_eval_images"]`), so it still produces
metrics instead of being skipped.

In [ ]:
# No runtime restart needed after a git pull: reload the changed modules IN PLACE,
# in dependency order. Reload `concepts` BEFORE `runner` — runner imports concepts, so
# reloading runner second re-binds it to the new concepts code. (config is NOT reloaded:
# that would swap out the CONFIG object this notebook already holds.)
import importlib, concepts as concept_mod, runner, viz
importlib.reload(concept_mod)
importlib.reload(runner)
importlib.reload(viz)

# Run every grade (or e.g. runner.run_all(["3_severe_npdr", "4_proliferative_dr"]) for a
# subset). A grade whose concepts come up short (or that errors otherwise) is SKIPPED, not
# fatal: `failures` maps each skipped grade -> reason and is printed as a numbered list below.
results, agg, failures = runner.run_all()

import pandas as pd
print("Per-method mean over", len(results), "grades (Acc, C-Ins):")
display(pd.DataFrame(agg).T)

# Per-grade breakdown: one row per DR grade (+ an Average row), one column per method, for
# each metric. Shown as a DataFrame and saved as a table image (best value bold / second-best
# underlined) via viz.render_metric_table.
methods = ("LGMD", "FACE", "ICE", "CRAFT")
for metric in ("Acc", "C-Ins"):
    tbl = runner.metric_table(results, metric, methods=methods)
    print(f"\nPer-grade {metric} (rows = grades):")
    display(pd.DataFrame(tbl[bb]).T.reindex(columns=list(methods)))
    viz.render_metric_table(
        tbl, out_name=f"table_pergrade_{metric.lower().replace('-', '')}_{bb}.png",
        methods=methods, backbones=(bb,), higher_is_better=True,
        title=f"Per-grade {metric} — {bb}")

In [ ]:
# Per-class performance across ALL metrics used, one table per retinal-disease class
# (methods as rows; Acc / C-Ins / agreement are higher-better, kl / recon_err lower-better).
# Best method per metric is bold, second-best underlined; each table is saved to viz_dir as
# table_<class>_metrics.png.
import pandas as pd
metrics_all = ["Acc", "C-Ins", "agreement", "kl", "recon_err"]
for cls, r in results.items():
    comp = r.get("comparison")
    if not comp:
        continue
    print(f"\n=== {cls}: methods x metrics ===")
    display(pd.DataFrame(comp).T.reindex(index=list(methods), columns=metrics_all))
    viz.render_class_metric_table(cls, comp, metrics=metrics_all, methods=methods,
                                  out_name=f"table_{cls.lower()}_metrics.png")

## 14. Concept grounding (FLAIR) — table + figure

Independent of the basis fit, measure how well FLAIR visually grounds each concept via the
localized similarity maps S: **peak** (strongest localized cell, averaged over images) and
**mean** (diffuse presence). No concept is dropped — this is a diagnostic. It emits a printed
per-grade table, a JSON (`results/concept_grounding_<run_tag>.json`), and a sorted bar/heatmap
figure. It needs only FLAIR + the train images (cached S), so it runs without the trained head.

In [ ]:
import importlib, runner
importlib.reload(runner)

# Prints a per-grade table, saves JSON, and renders a sorted bar/heatmap figure (peak).
# Use grounding_table(score="mean") to sort/color by the diffuse mean instead.
grounding = runner.grounding_table()

### Recover a skipped grade
A grade is only skipped if it has **no concepts** listed for it (or it errored for another
reason). To recover: add a few 2–3 word clinical concepts for that grade directly in
[`concept_vocab.json`](concept_vocab.json) (under its `class_<grade>` key), then re-evaluate
**only** that grade — every grade that already succeeded stays cached. Editing the vocab file
changes its content hash, so the concept/S/W/metrics caches for the edited grade rebuild
automatically.

In [ ]:
# === Recover a skipped grade: edit concept_vocab.json above, then re-evaluate it here ===
# After adding concepts for the grade in concept_vocab.json, set `grade` to its folder name
# and re-run just that grade. config._vocab_hash() is recomputed on import, but this kernel
# already holds the old hash, so refresh it so the concept caches invalidate:
import importlib, config, runner, concepts as concept_mod
importlib.reload(concept_mod); importlib.reload(runner)
config.CONFIG["concept_vocab_hash"] = config._vocab_hash()   # pick up the edited vocab file

grade = "1_mild_npdr"                             # <- the skipped grade to re-evaluate
res, _, refail = runner.run_all([grade])         # re-evaluate just this grade
results.update(res)                              # merge into the full results
failures.pop(grade, None)
failures.update(refail)                          # still failing? it stays for another pass

import pandas as pd
print("Remaining skipped grades:", list(failures))
print("Per-method mean over", len(results), "grades (Acc, C-Ins):")
display(pd.DataFrame(runner.aggregate(results)).T)

## Qualitative concept figures (grades 2/3/4)

Render the concept-overlay figures for the lesion-rich grades. `make_figures` defaults to
`config.FIGURE_CLASSES` (grades 2/3/4); pass an explicit list to change the set/order.
Artifacts are cached per grade, so reruns are cheap.

In [ ]:
import importlib, runner, config
importlib.reload(runner)

# Concept figures for the showcase grades (default config.FIGURE_CLASSES = grades 2/3/4).
# Pass a list to restrict/reorder, e.g. make_figures(["4_proliferative_dr", "3_severe_npdr"]).
fig_data = runner.make_figures()